### Data analysis
Reads all CSVs from `results/experiment_id`, concatenates them, saves under `processed_results/experiment_id`, computes mean features per well, and plots area in plate view.

### Imports

In [ ]:
import os
from pathlib import Path
import pandas as pd
from utils_data_analysis import extract_well_id_and_timepoint, plot_plate_view

### Config

In [ ]:
# Experiment ID (must match the folder name under results/)
experiment_id = "260728_SK_SK0075_Exp03_ConfocalMic"

results_dir = Path("results") / experiment_id
processed_dir = Path("processed_results") / experiment_id
os.makedirs(processed_dir, exist_ok=True)
print(f"Reading from {results_dir}")
print(f"Writing to {processed_dir}")

### Read all CSVs, concatenate, and save

In [ ]:
csv_files = sorted([f for f in results_dir.glob("*.csv") if f.name != "infection_summary.csv"])
if not csv_files:
    raise FileNotFoundError(f"No CSV files in {results_dir}")

df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

df_all = extract_well_id_and_timepoint(df_all)

out_path = processed_dir / "concatenated.csv"
df_all.to_csv(out_path, index=False)
print(f"Concatenated {len(csv_files)} files → {len(df_all)} rows. Saved to {out_path}")

In [ ]:
df_all

### Mean features per well and plate view of feature

In [ ]:
# Average all numeric features by well_id
df_mean = df_all.groupby(["well_id", "timepoint"], as_index=False).mean(numeric_only=True)

# Save mean dataframe
mean_path = processed_dir / "mean_per_well.csv"
df_mean.to_csv(mean_path, index=False)
print(f"Mean per well saved to {mean_path}")

In [ ]:
# Create a directory for plate view plots if doesn't exist
plate_view_dir = processed_dir / "plate_view"
plate_view_dir.mkdir(exist_ok=True)

for timepoint in df_mean["timepoint"].unique():
    df_time = df_mean[df_mean["timepoint"] == timepoint]
    for feature in df_mean.columns:
        if feature in ['well_id', 'label', 'timepoint']:
            continue
        # Create filename append for this timepoint
        timepoint_str = f"{int(timepoint)}h"
        plot_plate_view(
            df_time.copy(),
            column_name=feature,
            title=f"{feature} ({timepoint_str})",
            label=feature,
            save_dir=str(plate_view_dir),
            fmt=0,
            display=False,
            save_name=f"{feature}_{timepoint_str}.png"
        )

### Plate view of % infected cells (infection_summary)

In [ ]:
infection_summary_path = results_dir / "infection_summary.csv"
df_inf = pd.read_csv(infection_summary_path)
df_inf = extract_well_id_and_timepoint(df_inf)

plate_view_dir = processed_dir / "plate_view"
plate_view_dir.mkdir(exist_ok=True)

for timepoint in sorted(df_inf["timepoint"].dropna().unique()):
    df_time = df_inf[df_inf["timepoint"] == timepoint]
    timepoint_str = f"{int(timepoint)}h"
    plot_plate_view(
        df_time.copy(),
        column_name="%_inf_cells",
        title=f"% infected cells ({timepoint_str})",
        label="%_inf_cells",
        save_dir=str(plate_view_dir),
        fmt=1,
        display=False,
        save_name=f"%_inf_cells_{timepoint_str}.png",
    )